# Búsqueda secuencial

El bucle for examina cada posición del vector secuencialmente. Si el elemento actual coincide con target, guardamos el índice y usamos break para no desperdiciar ciclos de CPU iterando el resto del arreglo.

In [ ]:
#include <iostream>
#include <vector>

int squentialSearch() {
    // Inicializamos un vector desordenado
    std::vector<int> numbers = {21, 12, 46, 2};
    int target = 46;
    int position = -1; // Usamos -1 para indicar que no se ha encontrado

    // Principio de la búsqueda secuencial: iteración simple
    for (size_t i = 0; i < numbers.size(); ++i) {
        if (numbers[i] == target) {
            position = i;
            break; // Detenemos la búsqueda al encontrar el objetivo
        }
    }
    return position;
}

int position = squentialSearch();

if (position != -1) {
        std::cout << "Elemento encontrado en el índice: " << position << "\n";
    } else {
        std::cout << "Elemento no encontrado.\n";
}


## `std::find`

En C++ profesional no solemos reinventar la rueda ("simple looping might come to mind, but there's a more efficient and expressive way" ). La STL encapsula esta misma lógica:

```cpp

template<class InputIt, class T = typename std::iterator_traits<InputIt>::value_type>
constexpr InputIt find(InputIt first, InputIt last, const T& value) {
    for (; first != last; ++first)
        if (*first == value)
            return first;
 
    return last;
}

* ***Plantillas (Templates) e Iteradores***: `InputIt` representa un tipo de iterador. Cuando llamamos a la función con un vector, first y last reciben numbers.begin() y numbers.end() respectivamente.

* ***Bucle*** `for (; first != last; ++first)`: Aquí radica el enfoque directo y sin complicaciones ("no-frills approach") de la búsqueda lineal. El algoritmo verifica cada elemento desde el iterador de inicio, avanzando (`++first`) hasta llegar al límite final (last).

* ***Comparación*** `if (*first == value)`: Desreferenciamos el iterador (`*first`) para obtener el valor almacenado en esa posición de memoria y lo comparamos con nuestro objetivo (`value`).

* ***Retornos*** (`return first;` o `return last;`): Si el elemento existe, la función retorna el iterador apuntando a su ubicación exacta; de lo contrario, al finalizar el bucle, retorna el iterador last (que en la práctica es `numbers.end()`).

In [ ]:
#include <iterator> // Necesario para std::iterator_traits

// 1. Definición de la plantilla (Template) de búsqueda secuencial
template<class InputIt, class T = typename std::iterator_traits<InputIt>::value_type>
constexpr InputIt custom_find(InputIt first, InputIt last, const T& value) {
    // Recorremos el contenedor desde el iterador 'first' hasta 'last'
    while (first != last) {
        // Desreferenciamos el iterador para comparar el valor
        if (*first == value) {
            return first; // Retornamos el iterador si encontramos una coincidencia
        }
        ++first;
    }
    return last; // Si terminamos el bucle sin éxito, retornamos el límite final
}


// 2. Inicialización de los datos
std::vector<int> numbers_ = {21, 12, 46, 2};
int target = 46;

std::cout << "Iniciando busqueda lineal del valor: " << target << "\n";

// 3. Invocación de nuestra plantilla
auto iter = custom_find(numbers_.begin(), numbers_.end(), target);

// 4. Evaluación del resultado
if (iter != numbers_.end()) {
    // auto index = std::distance(numbers_.begin(), iter);
    auto index = (iter - numbers_.begin());
    std::cout << "Elemento encontrado en el indice: " << index << "\n";
} else {
    std::cout << "Elemento no encontrado.\n";
}

* `template<class InputIt...>` y `custom_find(...)`: Esta plantilla permite que nuestra función trabaje con cualquier contenedor que admita iteradores (como `std::vector` o las listas doblemente enlazadas). El tipo `T` deduce automáticamente el tipo de dato que almacena el contenedor (en este caso, int).

* `while (first != last)`: Este es el núcleo de la búsqueda lineal, el cual es el algoritmo de búsqueda más básico e intuitivo. Comienza en el iterador de inicio y avanza elemento por elemento.

* `if (*first == value)`: Comprobamos cada elemento desde el principio hasta el final. Al usar el operador `*` (desreferenciación), accedemos al valor real almacenado en la memoria a la que apunta el iterador.

* `return first;` vs `return last;`: Si el elemento existe, la función devolverá un iterador apuntando a su ubicación. Si el bucle termina sin encontrar el target, devolverá el iterador last (que corresponde a `numbers_.end()`).

* `auto it = custom_find(...)` en el main: Pasamos los iteradores `begin()` y `end()` de nuestro vector. Gracias a la deducción de tipos de C++ (auto), el compilador sabe que it será un iterador de un vector de enteros.

* `auto index = std::distance(numbers_.begin(), iter);`: Como los iteradores de std::vector son de acceso aleatorio y la memoria es contigua, podemos restar el iterador base al iterador devuelto para obtener el índice exacto (0-indexado) donde se encuentra nuestro elemento. `find` no entiende índices. distance es el traductor que convierte una posición (iterador) en un número (índice) solo si el usuario humano lo necesita para imprimirlo en pantalla.

In [ ]:
#include <algorithm>   // Para el uso de algorithms 

std::vector<int> numbers = {21, 12, 46, 2}; //
    
// Utilizamos std::find para abstraer el bucle for
auto it = std::find(numbers.begin(), numbers.end(), 46); // 
    
// Si el elemento existe, iterador apunta a su ubicación; si no, a numbers.end()
if (it != numbers.end()) { // 
    std::cout << "Elemento encontrado en el índice: " << (it - numbers.begin()) << "\n";
} else {
    std::cout << "Elemento no encontrado.\n";
}

este método es ideal porque funciona en vectores ordenados y desordenados. Sin embargo, el tiempo que toma crece linealmente con el tamaño del vector, haciéndolo menos ideal para conjuntos de datos masivos.  ***¿Cómo podemos buscar más rápido si garantizamos que nuestros datos están ordenados?***


## `std::lower_bound(...)` y `std::upper_bound(...)`

# Búsqueda binaria

In [ ]:
// Implementación manual de la Búsqueda Binaria (Under the Hood)
bool custom_binary_search(const std::vector<int>& dataset, int target) {
    int left = 0;
    int right = dataset.size() - 1;

    // Continuamos mientras el espacio de búsqueda sea válido
    while (left <= right) {
        // 1. "Hace un viaje directo al centro del conjunto de datos"
        int mid = left + (right - left) / 2; 

        // 2. Comprobamos si el elemento del centro es nuestro objetivo
        if (dataset[mid] == target) {
            return true; // ¡Encontrado!
        }

        // 3. "Determina si el elemento deseado se encuentra en la primera o segunda mitad"
        if (dataset[mid] < target) {
            // El objetivo es mayor, "descarta la mitad" izquierda
            left = mid + 1;
        } else {
            // El objetivo es menor, "descarta la mitad" derecha
            right = mid - 1;
        }
    }

    // Si el campo de búsqueda se reduce a cero y no lo encontramos
    return false;
}


// El prerrequisito innegociable: el vector debe estar ordenado
std::vector<int> sorted_numbers = {10, 20, 30, 40, 50, 60, 70};
target = 60;

std::cout << "Iniciando busqueda binaria manual del valor: " << target << "\n";

bool found = custom_binary_search(sorted_numbers, target);

if (found) {
    std::cout << "¡Elemento encontrado exitosamente!\n";
} else {
    std::cout << "Elemento no encontrado.\n";
}

* `int left = 0; int right = dataset.size() - 1;`: Definimos nuestros límites de búsqueda iniciales, abarcando todo el vector.
  
* `int mid = left + (right - left) / 2;`: Esta es la traducción matemática de "hacer un viaje directo al centro del conjunto de datos". (Nota técnica: escribimos `left + (right - left) / 2` en lugar de `(left + right) / 2` para evitar desbordamientos de enteros en arreglos masivos).

* `if (dataset[mid] == target)`: El mejor escenario posible. ¡Acertamos al elemento exacto en el medio!

* `if (dataset[mid] < target)` y el bloque `else`: Hacemos una evaluación rápida para saber en qué mitad está el dato. Si el valor central es menor que nuestro objetivo, sabemos con certeza (porque el arreglo está ordenado ) que el objetivo debe estar a la derecha. Por lo tanto, movemos nuestro límite `left` a `mid + 1`, descartando efectivamente la mitad izquierda completa de los elementos restantes.

* El bucle `while (left <= right)`: Representa el proceso de "reducir continuamente el campo de búsqueda". La complejidad de tiempo es logarítmica ($O(\log n)$) precisamente porque dividimos el tamaño del problema por 2 en cada iteración.

In [ ]:
// Se requiere #include <algorithm>
// Es imperativo que el vector esté ordenado para usar búsqueda binaria
std::vector<int> numbers2 = {1, 3, 3, 5, 7}; // 

// --- Uso de std::lower_bound ---
int val1 = 3; 
auto low1 = std::lower_bound(numbers2.begin(), numbers2.end(), val1); 
std::cout << "std::lower_bound para el valor " << val1 
              << ": " << (low1 - numbers2.begin()) << "\n"; 

int val2 = 4; 
auto low2 = std::lower_bound(numbers2.begin(), numbers2.end(), val2); 
std::cout << "std::lower_bound para el valor " << val2 
              << ": " << (low2 - numbers2.begin()) << "\n"; 

// --- Uso de std::upper_bound ---
int val3 = 3; 
auto up1 = std::upper_bound(numbers2.begin(), numbers2.end(), val3); 
std::cout << "std::upper_bound para el valor " << val3 
              << ": " << (up1 - numbers2.begin()) << "\n"; 

* `std::vector<int> numbers = {1, 3, 3, 5, 7};`: El vector ya se encuentra ordenado. Si no lo estuviera, los resultados de la búsqueda binaria serían impredecibles.
  
* `std::lower_bound` ***(Límite Inferior)***: Esta función retorna un iterador apuntando al primer elemento que no es menor que (es decir, mayor o igual a) el valor especificado.Para `val1 = 3`, retorna un iterador que apunta a la primera aparición del 3, que se encuentra en el índice 1. Para `val2 = 4` (un valor que no existe en el vector), la función nos indica dónde encajaría mejor manteniendo la naturaleza ordenada del vector. Nos señala la posición justo antes del 5, es decir, el índice 3.

* `std::upper_bound` ***(Límite Superior)***: Identifica la primera posición donde reside un elemento que es estrictamente mayor que el valor especificado.Para val3 = 3, dado que tenemos múltiples ocurrencias, std::upper_bound apuntará justo después de la última aparición del 3. Esto es justo antes del 5, en el índice 3.

* ***Aritmética de iteradores***: Usamos (`low1 - numbers.begin()`) para calcular el índice numérico a partir del iterador devuelto.

# Resumen

In [ ]:
// El vector debe estar ordenado para que la búsqueda binaria funcione
std::vector<int> data = {10, 20, 30, 30, 40, 50}; 
target = 30;

std::cout << "--- Busqueda en C++ STL ---\n\n";

// --- 1. Búsqueda Lineal (Secuencial) ---
// Enfoque directo que comprueba cada elemento desde el principio hasta el final
auto it_linear = std::find(data.begin(), data.end(), target); 
    
if (it_linear != data.end()) {
    std::cout << "[Lineal] std::find encontro el valor " << target 
    << " en el indice: " << (it_linear - data.begin()) << "\n";
}

// --- 2. Búsqueda Binaria ---
// Reduce continuamente el campo de búsqueda a la mitad
std::cout << "\nResultados de Busqueda Binaria:\n";

// a) Confirmación rápida y estricta de existencia (Retorna bool)
bool exists = std::binary_search(data.begin(), data.end(), target);
std::cout << "- std::binary_search confirmo su existencia: " 
          << (exists ? "Si (True)" : "No (False)") << "\n";
    
// b) Encontrar el inicio (primer elemento no menor que el valor)
auto it_lower = std::lower_bound(data.begin(), data.end(), target); 
std::cout << "- std::lower_bound apunta al indice: " 
    << (it_lower - data.begin()) << "\n";
    
// c) Encontrar el final (primer elemento mayor que el valor)
auto it_upper = std::upper_bound(data.begin(), data.end(), target); 
std::cout << "- std::upper_bound apunta al indice: " 
     << (it_upper - data.begin()) << "\n";

// d) Calcular cuántos elementos repetidos hay usando los iteradores
// Si tenemos múltiples ocurrencias de un elemento, upper_bound apuntará justo pasada la última ocurrencia [cite: 195]
std::cout << "- Cantidad de veces que aparece el " << target 
     << ": " << (it_upper - it_lower) << " vez/veces.\n";
